In [3]:
# ==============================================================================
# CBU TAAS Diagnostics ReAct AI Agent
# Model: Qwen/Qwen2.5-72B-Instruct via Hugging Face Inference API
# Features: Explicit ReAct Loop, Retry Mechanism (429/503), Robust Tool Matching
# ==============================================================================

!pip install -q -U "huggingface-hub<2.0" tenacity

import json
import re
from huggingface_hub import InferenceClient
from google.colab import userdata
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# --- 1. INITIALIZATION & CLIENT SETUP ---
HF_TOKEN = userdata.get('HF_TOKEN')
client = InferenceClient(api_key=HF_TOKEN)

MODEL_ID = "Qwen/Qwen2.5-72B-Instruct"


# --- 2. PARAMETERIZED DIAGNOSTIC TOOLS (WITH ROBUST PATH NORMALIZATION) ---
def get_test_logs(test_id: str) -> str:
    """Fetch execution logs and failure traces for a given test case ID."""
    mock_logs = {
        "TC_BILLING_9901": {
            "test_name": "test_postpaid_billing_retry",
            "status": "FAILED",
            "error": "TimeoutError: Request to payment gateway timed out after 5000ms",
            "trace": "Playwright / API Call failed on Endpoint POST /api/v1/charge"
        },
        "TC_PLAYWRIGHT_102": {
            "test_name": "test_subscriber_dashboard_render",
            "status": "FLAKY",
            "error": "ElementHandle.click: Timeout 3000ms exceeded while waiting for element #submit-btn",
            "trace": "DOM hydration delay caused race condition on slow network profile"
        }
    }
    return json.dumps(mock_logs.get(test_id.strip(), {"error": f"Test ID '{test_id}' not found"}))

def check_endpoint_status(endpoint: str) -> str:
    """Check network status, latency, and circuit breaker status for an API endpoint."""
    # Strip HTTP methods (POST, GET, etc.) and extra spaces
    clean_ep = re.sub(r'^(POST|GET|PUT|DELETE|PATCH)\s+', '', endpoint.strip(), flags=re.IGNORECASE)

    mock_endpoints = {
        "/api/v1/charge": {
            "status": "DEGRADED",
            "latency_ms": 5200,
            "active_rate_limit": True,
            "circuit_breaker": "HALF-OPEN"
        },
        "/dashboard/subscriber": {
            "status": "HEALTHY",
            "latency_ms": 320,
            "active_rate_limit": False,
            "circuit_breaker": "CLOSED"
        }
    }

    # Fuzzy match logic for path aliases (e.g., subscriber_dashboard -> /dashboard/subscriber)
    if clean_ep in mock_endpoints:
        return json.dumps(mock_endpoints[clean_ep])

    for path, data in mock_endpoints.items():
        if path in clean_ep or clean_ep in path or ("dashboard" in clean_ep and "dashboard" in path):
            return json.dumps(data)

    return json.dumps({"error": f"Endpoint '{endpoint}' not found"})

def inspect_database_metrics(metric_type: str) -> str:
    """Inspect backend database metrics and lock contention levels."""
    metrics = {
        "all": {
            "pending_transactions": 1420,
            "lock_contention": "HIGH",
            "db_cpu_utilization": "94%"
        },
        "connection_pool": {
            "active_connections": 98,
            "max_connections": 100,
            "waiting_queries": 45
        }
    }
    return json.dumps(metrics.get(metric_type.strip(), metrics["all"]))

TOOLS_MAP = {
    "get_test_logs": get_test_logs,
    "check_endpoint_status": check_endpoint_status,
    "inspect_database_metrics": inspect_database_metrics
}


# --- 3. REACT ENGINE WITH EXPONENTIAL BACKOFF RETRY ---
SYSTEM_PROMPT = """You are an expert CBU TAAS Diagnostics AI Agent.
Analyze automated test failures step-by-step using a strict ReAct (Reasoning + Acting) loop.

Available Tools:
- get_test_logs(test_id="...")
- check_endpoint_status(endpoint="...")
- inspect_database_metrics(metric_type="...")

You MUST strictly output every turn using this exact format:

Thought: <Explain step-by-step reasoning on what to check next>
Action: <tool_name>(<argument_value>)

When you have collected all required observations, conclude with:

Thought: <Final concluding summary of findings>
Answer:
=== TAAS TEST DIAGNOSTIC REPORT ===
Test Case ID: <ID>
Failure Type: <Type>
Root Cause: <Root Cause Description>
Remediation: <Remediation Steps>
"""

class ReActAgentHF:
    def __init__(self, client, model_id: str, tools_map: dict):
        self.client = client
        self.model_id = model_id
        self.tools_map = tools_map

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type(Exception),
        reraise=True
    )
    def _call_hf_with_retry(self, messages: list) -> str:
        response = self.client.chat_completion(
            model=self.model_id,
            messages=messages,
            max_tokens=600,
            temperature=0.1
        )
        return response.choices[0].message.content

    def run(self, user_query: str, max_steps: int = 5):
        print(f"============================================================")
        print(f" STARTING REACT AGENT RUN ({self.model_id})")
        print(f" QUERY: {user_query}")
        print(f"============================================================\n")

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_query}
        ]

        for step_idx in range(1, max_steps + 1):
            try:
                output_text = self._call_hf_with_retry(messages)
            except Exception as e:
                print(f"[API RETRY FAILED] Unable to communicate with HF API: {e}")
                break

            print(f"--- STEP {step_idx} ---")
            print(output_text)
            messages.append({"role": "assistant", "content": output_text})

            if "Answer:" in output_text:
                print("\n[SUCCESS] ReAct Diagnostic Loop Completed.")
                break

            # Parse Action step using regex
            action_match = re.search(r"Action:\s*(\w+)\((.*?)\)", output_text)
            if action_match:
                tool_name = action_match.group(1)
                raw_arg = action_match.group(2).strip("'\" ")

                if "=" in raw_arg:
                    raw_arg = raw_arg.split("=")[1].strip("'\" ")

                if tool_name in self.tools_map:
                    try:
                        observation = self.tools_map[tool_name](raw_arg)
                    except Exception as err:
                        observation = f"Tool execution error: {str(err)}"
                else:
                    observation = f"Error: Tool '{tool_name}' not available."

                obs_str = f"Observation: {observation}"
                print(f"\n{obs_str}\n" + "-" * 60)
                messages.append({"role": "user", "content": obs_str})
            else:
                print("\n[Warning] No structured Action found. Terminating loop.")
                break


# --- 4. EXECUTION & SCENARIO TESTING ---
agent = ReActAgentHF(client=client, model_id=MODEL_ID, tools_map=TOOLS_MAP)

# Scenario A: Intermittent API Timeout in Telecom Billing Suite
agent.run("Diagnose the failure in CBU Telecom Billing test suite for TC_BILLING_9901.")

print("\n" + "="*80 + "\n")

# Scenario B: Flaky Playwright UI Test Suite
agent.run("Diagnose the intermittent issue for Playwright UI test case TC_PLAYWRIGHT_102.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 18.4 MB/s eta 0:00:00
 STARTING REACT AGENT RUN (Qwen/Qwen2.5-72B-Instruct)
 QUERY: Diagnose the failure in CBU Telecom Billing test suite for TC_BILLING_9901.

--- STEP 1 ---
Thought: The first step is to gather the test logs for the specific test case ID to understand the nature of the failure.
Action: get_test_logs(test_id="TC_BILLING_9901")

Observation: {"test_name": "test_postpaid_billing_retry", "status": "FAILED", "error": "TimeoutError: Request to payment gateway timed out after 5000ms", "trace": "Playwright / API Call failed on Endpoint POST /api/v1/charge"}
------------------------------------------------------------
--- STEP 2 ---
Thought: The test logs indicate a timeout error when making a request to the payment gateway. This could be due to a network issue, a problem with the payment gateway itself, or an issue with the API endpoint. The next step is to check the status of the payment gateway endpoint.
Action: ch